# AdaDetect Benchmark Notebook (GPU + TRUE CNN)

## Overview
This notebook benchmarks AdaDetect with various classifiers including a GPU-accelerated CNN and classical ML models.

## 1. INSTALL (run once)
```bash
pip install medmnist torchvision
```

## 2. IMPORTS

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.models as models

from torchvision import datasets, transforms
import medmnist
from medmnist import INFO

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

from procedure import AdaDetectERM

np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cudnn.benchmark = True
print(f"Using device: {device}")

## 3. DATA LOADERS (KEEP IMAGE SHAPE)

In [ ]:
def load_mnist():
    dataset = datasets.MNIST(root="./data", train=True, download=True)

    X = dataset.data.numpy() / 255.0  # (N, 28, 28)
    X = np.expand_dims(X, 1)          # (N, 1, 28, 28)
    y = dataset.targets.numpy()

    return X, y


def load_medmnist(name):
    info = INFO[name]
    DataClass = getattr(medmnist, info['python_class'])

    dataset = DataClass(split='train', download=True)

    X = dataset.imgs / 255.0

    if len(X.shape) == 3:
        X = np.expand_dims(X, 1)
    elif X.shape[-1] == 3:
        X = np.transpose(X, (0, 3, 1, 2))

    y = dataset.labels

    # =========================
    # SPECIAL CASE: ChestMNIST
    # benign vs non-bengin
    # =========================
    if name == "chestmnist":
        # any disease present => non-benign (1)
        # all zeros => benign (0)
        y = (y.sum(axis=1) > 0).astype(int)

    # =========================
    # Other datasets
    # =========================
    elif len(y.shape) > 1 and y.shape[1] > 1:
        y = (y.sum(axis=1) > 0).astype(int)
    else:
        y = y.squeeze()

    return X, y


def get_datasets():
    return {
        "mnist": load_mnist(),
        "derma_mnist": load_medmnist("dermamnist"),
        "chest_mnist": load_medmnist("chestmnist"),
    }

print("Data loaders defined")

## 4. SPLIT

In [ ]:
def split_null_signal(X, y, null_class=0):
    mask_null = (y == null_class)
    return X, X[mask_null], ~mask_null

## 5. TRUE CNN CLASSIFIER (GPU)

In [ ]:
class CNNClassifier:
    def __init__(self, in_channels=1, epochs=5, lr=1e-3, batch_size=128):
        self.epochs = epochs
        self.lr = lr
        self.batch_size = batch_size
        self.in_channels = in_channels

        self.model = nn.Sequential(
            nn.Conv2d(in_channels, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        ).to(device)

    def fit(self, X, y):
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.long)

        loader = torch.utils.data.DataLoader(
            torch.utils.data.TensorDataset(X, y),
            batch_size=self.batch_size,
            shuffle=True,
            pin_memory=True
        )

        opt = optim.Adam(self.model.parameters(), lr=self.lr)
        loss_fn = nn.CrossEntropyLoss()

        self.model.train()
        for _ in range(self.epochs):
            for xb, yb in loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                opt.zero_grad()
                loss = loss_fn(self.model(xb), yb)
                loss.backward()
                opt.step()

    def predict_proba(self, X):
        X = torch.tensor(X, dtype=torch.float32)

        loader = torch.utils.data.DataLoader(X, batch_size=self.batch_size)

        self.model.eval()
        out = []

        with torch.no_grad():
            for xb in loader:
                xb = xb.to(device)
                out.append(torch.softmax(self.model(xb), dim=1).cpu())

        return torch.cat(out).numpy()

## RESNET

In [ ]:
class ResNetClassifier:
    def __init__(self, in_channels=1, lr=1e-3, epochs=5, batch_size=128):
        self.in_channels = in_channels
        self.lr = lr
        self.epochs = epochs
        self.batch_size = batch_size

        # ---- Load pretrained ResNet18 ----
        self.model = models.resnet18(weights=None)

        # ---- Replace first conv layer for arbitrary channels ----
        self.model.conv1 = nn.Conv2d(
            in_channels,
            64,
            kernel_size=7,
            stride=2,
            padding=3,
            bias=False
        )

        # ---- Replace classifier head for binary task ----
        self.model.fc = nn.Linear(self.model.fc.in_features, 2)

        self.model = self.model.to(device)

    def fit(self, X, y):
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.long)

        loader = torch.utils.data.DataLoader(
            torch.utils.data.TensorDataset(X, y),
            batch_size=self.batch_size,
            shuffle=True,
            pin_memory=True
        )

        optimizer = optim.Adam(self.model.parameters(), lr=self.lr)
        loss_fn = nn.CrossEntropyLoss()

        self.model.train()

        for _ in range(self.epochs):
            for xb, yb in loader:
                xb = xb.to(device, non_blocking=True)
                yb = yb.to(device, non_blocking=True)

                optimizer.zero_grad()
                loss = loss_fn(self.model(xb), yb)
                loss.backward()
                optimizer.step()

    def predict_proba(self, X):
        X = torch.tensor(X, dtype=torch.float32)

        loader = torch.utils.data.DataLoader(X, batch_size=self.batch_size)

        self.model.eval()
        outs = []

        with torch.no_grad():
            for xb in loader:
                xb = xb.to(device)
                outs.append(torch.softmax(self.model(xb), dim=1).cpu())

        return torch.cat(outs).numpy()

## 6. TABULAR CLASSIFIERS (FLATTENED)

In [ ]:
def flatten(X):
    return X.reshape(len(X), -1)


def get_classifiers():
    return {
        "knn": KNeighborsClassifier(3),
        "ResNet": ResNetClassifier(),
        "CNN": CNNClassifier(),
        # "svm": SVC(probability=True),
        "decision_tree": DecisionTreeClassifier(max_depth=5),
        "random_forest": RandomForestClassifier(n_estimators=100),
        "mlp": MLPClassifier(max_iter=300),
        "adaboost": AdaBoostClassifier(),
        "naive_bayes": GaussianNB(),
        # "gaussian_process": GaussianProcessClassifier(1.0 * RBF(1.0)),
    }

print("Classifiers defined")

## 7. METRICS

In [ ]:
def compute_fdr_power(rejection_set, is_signal):
    R = len(rejection_set)
    if R == 0:
        return 0.0, 0.0

    rejected = np.zeros_like(is_signal, dtype=bool)
    rejected[rejection_set] = True

    V = np.sum(rejected & ~is_signal)
    S = np.sum(rejected & is_signal)

    return V / R, S / np.sum(is_signal)

## 8. RUN ADaDETECT

In [ ]:
def run_adadetect(model, x, xnull, is_signal, flatten_data=False):

    if flatten_data:
        x = flatten(x)
        xnull = flatten(xnull)

    proc = AdaDetectERM(scoring_fn=model, split_size=0.5)
    rejection_set = proc.apply(x=x, level=0.1, xnull=xnull)

    return compute_fdr_power(rejection_set, is_signal)

## 9. BENCHMARK

In [ ]:
def build_model(name, in_channels):
    if name == "cnn":
        return CNNClassifier(in_channels=in_channels)

    if name == "resnet":
        return ResNetClassifier(in_channels=in_channels)

    if name == "svm":
        return SVC(probability=True)

    if name == "rf":
        return RandomForestClassifier(n_estimators=100)

    if name == "knn":
        return KNeighborsClassifier(3)

    if name == "mlp":
        return MLPClassifier(max_iter=300)

    if name == "adaboost":
        return AdaBoostClassifier()

    if name == "nb":
        return GaussianNB()

    if name == "gp":
        return GaussianProcessClassifier(1.0 * RBF(1.0))

    raise ValueError(f"Unknown model: {name}")

In [ ]:
def benchmark():
    datasets = get_datasets()
    results = []

    model_names = [
        "resnet",
        "cnn",
        # "svm",
        "rf",
        "knn",
        "mlp",
        "adaboost",
        "nb",
        # "gp"
    ]

    for dname, (X, y) in datasets.items():
        print("\n" + "=" * 60)
        print(f"DATASET: {dname}")
        print("=" * 60)

        x, xnull, is_signal = split_null_signal(X, y)

        idx = np.random.choice(len(x), 4000, replace=False)
        x, is_signal = x[idx], is_signal[idx]

        idx_null = np.random.choice(len(xnull), min(4000, len(xnull)), replace=False)
        xnull = xnull[idx_null]

        print(f"Signal: {np.sum(is_signal)} | Null: {len(xnull)}")

        for name in model_names:
            print(f"\n[{name.upper()}]")

            try:
                model = build_model(name, in_channels=x.shape[1])

                flatten_flag = False if name in ["cnn", "resnet"] else True

                fdr, power = run_adadetect(
                    model,
                    x,
                    xnull,
                    is_signal,
                    flatten_data=flatten_flag
                )

                print(f"{name} -> FDR: {fdr:.4f}, Power: {power:.4f}")

                results.append({
                    "dataset": dname,
                    "model": name,
                    "fdr": fdr,
                    "power": power
                })

            except Exception as e:
                print(f"{name} failed:", e)

    return pd.DataFrame(results)

## 10. ANALYSIS

In [ ]:
def analyze(df):
    print(df.groupby(["dataset", "model"])[["fdr", "power"]].mean())

    for d in df.dataset.unique():
        sub = df[df.dataset == d]

        plt.figure()
        plt.scatter(sub.fdr, sub.power)

        for _, r in sub.iterrows():
            plt.text(r.fdr, r.power, r.model)

        plt.xlabel("FDR")
        plt.ylabel("Power")
        plt.title(d)
        plt.show()

## 11. INTERACTIVE EXECUTION CELLS
### Run ONE dataset at a time

In [ ]:
def prepare_dataset(name):
    datasets = get_datasets()
    X, y = datasets[name]

    print(f"\n=== Dataset: {name} ===")
    print("Total samples:", len(y))
    print("Benign (0):", np.sum(y == 0))
    print("Non-benign (1):", np.sum(y == 1))

    x, xnull, is_signal = split_null_signal(X, y)

    # subsample for speed
    idx = np.random.choice(len(x), 4000, replace=False)
    x, is_signal = x[idx], is_signal[idx]

    idx_null = np.random.choice(len(xnull), min(4000, len(xnull)), replace=False)
    xnull = xnull[idx_null]

    print("Signal samples after split:", np.sum(is_signal))
    print("Null samples:", len(xnull))

    return x, xnull, is_signal

In [ ]:
bench_df = benchmark()
analyze(bench_df)


### SELECT DATASET HERE
Uncomment one of the lines below to prepare a dataset:

In [ ]:
x, xnull, is_signal = prepare_dataset("mnist")
# x, xnull, is_signal = prepare_dataset("derma_mnist")
# x, xnull, is_signal = prepare_dataset("chest_mnist")

## 12. RUN SINGLE MODEL CELLS

### CNN (GPU)

In [ ]:
cnn = CNNClassifier(in_channels=x.shape[1])
fdr, power = run_adadetect(cnn, x, xnull, is_signal, flatten_data=False)
print("CNN -> FDR:", fdr, "Power:", power)

### RESNET (GPU)

In [ ]:
RN = ResNetClassifier(in_channels=x.shape[1])
fdr, power = run_adadetect(RN, x, xnull, is_signal, flatten_data=False)
print("ResNet -> FDR:", fdr, "Power:", power)

### SVM

In [ ]:
# svm = SVC(probability=True)
# fdr, power = run_adadetect(svm, x, xnull, is_signal, flatten_data=True)
# print("SVM -> FDR:", fdr, "Power:", power)

### Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100)
fdr, power = run_adadetect(rf, x, xnull, is_signal, flatten_data=True)
print("RF -> FDR:", fdr, "Power:", power)

### KNN

In [ ]:
knn = KNeighborsClassifier(3)
fdr, power = run_adadetect(knn, x, xnull, is_signal, flatten_data=True)
print("KNN -> FDR:", fdr, "Power:", power)

### MLP

In [ ]:
mlp = MLPClassifier(max_iter=300)
fdr, power = run_adadetect(mlp, x, xnull, is_signal, flatten_data=True)
print("MLP -> FDR:", fdr, "Power:", power)

### AdaBoost

In [ ]:
ada = AdaBoostClassifier()
fdr, power = run_adadetect(ada, x, xnull, is_signal, flatten_data=True)
print("AdaBoost -> FDR:", fdr, "Power:", power)

### Naive Bayes

In [ ]:
nb = GaussianNB()
fdr, power = run_adadetect(nb, x, xnull, is_signal, flatten_data=True)
print("NB -> FDR:", fdr, "Power:", power)

### Gaussian Process (SLOW)

In [ ]:
# gp = GaussianProcessClassifier(1.0 * RBF(1.0))
# fdr, power = run_adadetect(gp, x, xnull, is_signal, flatten_data=True)
# print("GP -> FDR:", fdr, "Power:", power)

## 13. PARALLEL EXECUTION (MULTIPLE MODELS)

In [ ]:
from joblib import Parallel, delayed


def run_model(name, model, x, xnull, is_signal, flatten_flag):
    try:
        fdr, power = run_adadetect(model, x, xnull, is_signal, flatten_data=flatten_flag)
        return {"model": name, "fdr": fdr, "power": power}
    except Exception as e:
        return {"model": name, "fdr": None, "power": None, "error": str(e)}


def run_all_models_parallel(x, xnull, is_signal, n_jobs=2):
    tasks = [
        ("cnn", CNNClassifier(), False),
        ("svm", SVC(probability=True), True),
        ("rf", RandomForestClassifier(n_estimators=100), True),
        ("knn", KNeighborsClassifier(3), True),
        ("mlp", MLPClassifier(max_iter=300), True),
        ("adaboost", AdaBoostClassifier(), True),
        ("naive_bayes", GaussianNB(), True),
        # ("gaussian_process", GaussianProcessClassifier(1.0 * RBF(1.0)), True),
    ]

    results = Parallel(n_jobs=n_jobs)(
        delayed(run_model)(name, model, x, xnull, is_signal, flatten_flag)
        for name, model, flatten_flag in tasks
    )

    return pd.DataFrame(results)

print("Parallel execution functions defined")

## 14. RUN PARALLEL
Uncomment to run parallel benchmarking:

In [ ]:
# # Example:
# x, xnull, is_signal = prepare_dataset("mnist")
# df_parallel = run_all_models_parallel(x, xnull, is_signal, n_jobs=2)
# print(df_parallel)

In [5]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from procedure import AdaDetectERM
def get_fdp(ytrue, rejection_set):
    if rejection_set.size:
        fdp = np.sum(ytrue[rejection_set] == 0) / len(rejection_set)
        tdp = np.sum(ytrue[rejection_set] == 1) / np.sum(ytrue == 1)
    else:
        fdp = 0
        tdp = 0
    return fdp, tdp
def sample_2d_pu(two_sided=True, seed=0):
    RNG = np.random.default_rng(seed)

    n = 5000
    m0 = 900
    m1 = 100
    m = m0 + m1

    Y = RNG.normal(loc=[0.0, 0.0], scale=[1.0, 1.0], size=(n, 2))

    X_0 = RNG.normal(loc=[0.0, 0.0], scale=[1.0, 1.0], size=(m0, 2))

    if two_sided:
        mix = RNG.binomial(1, 0.5, size=m1)
        means = np.where(mix[:, None] == 1, [2.0, 2.0], [-2.0, -2.0])
        X_1 = RNG.normal(loc=means, scale=[1.0, 1.0], size=(m1, 2))
    else:
        X_1 = RNG.normal(loc=[2.0, 2.0], scale=[1.0, 1.0], size=(m1, 2))

    Z = np.vstack([Y, X_0, X_1])

    y_true = np.concatenate([
        np.zeros(n, dtype=int),
        np.zeros(m0, dtype=int),
        np.ones(m1, dtype=int)
    ])

    return Z, y_true

In [6]:
def run_once_on_data(model, Z, y_true, n, m, k, level=0.05):

    xnull = Z[:n]
    x = Z[n:n+m]
    y_test_true = y_true[n:n+m]

    proc = AdaDetectERM(
        scoring_fn=model,
        correction_type="storey",
        split_size=k / n   # ✅ ORIGINAL STRUCTURE KEPT
    )

    rejection_set = proc.apply(x, level, xnull)

    if len(rejection_set):
        fdp = np.sum(y_test_true[rejection_set] == 0) / len(rejection_set)
        tdp = np.sum(y_test_true[rejection_set] == 1) / np.sum(y_test_true == 1)
    else:
        fdp, tdp = 0.0, 0.0

    return (fdp, tdp), len(rejection_set)

In [3]:
B = 5

fdps_logreg, tdps_logreg, rejs_logreg = [], [], []
fdps_nn, tdps_nn, rejs_nn = [], [], []

for b in range(B):

    Z, y_true, n, m, k = load_pu_synthetic(two_sided=True, seed=b)

    # Logistic Regression
    (fdp, tdp), r = run_pu_experiment(
        LogisticRegression(max_iter=2000),
        Z, y_true, n, m, k
    )

    fdps_logreg.append(fdp)
    tdps_logreg.append(tdp)
    rejs_logreg.append(r)

    # Neural Net
    (fdp, tdp), r = run_pu_experiment(
        MLPClassifier(hidden_layer_sizes=(50, 50),
                      max_iter=2000,
                      random_state=b),
        Z, y_true, n, m, k
    )

    fdps_nn.append(fdp)
    tdps_nn.append(tdp)
    rejs_nn.append(r)

In [4]:
print("Logistic regression:")
print("Mean FDP:", np.mean(fdps_logreg))
print("Mean TDP:", np.mean(tdps_logreg))
print("Mean rejections:", np.mean(rejs_logreg))

print("\nNeural network:")
print("Mean FDP:", np.mean(fdps_nn))
print("Mean TDP:", np.mean(tdps_nn))
print("Mean rejections:", np.mean(rejs_nn))

Logistic regression:
Mean FDP: 0.0
Mean TDP: 0.0
Mean rejections: 0.0

Neural network:
Mean FDP: 0.032329192546583854
Mean TDP: 0.242
Mean rejections: 25.6
